# Session 5 · Part 1 — Exploring Multispectral Satellite Data
## 🎓 STUDENT VERSION

**AI for Environmental Sustainability** · Dr. René Traoré, Eva Vormschlag, Prof. Marius Lindauer  
📦 Repository: https://github.com/automl-edu/AI_Environmental_Sustainability

---

## Learning objectives

1. Load and inspect multispectral satellite imagery (13-band Sentinel-2 via EuroSAT)
2. Visualise band composites and explain what each one reveals physically
3. Compute and interpret spectral indices (NDVI, NDWI, NDBI)
4. Run unsupervised k-means clustering on all bands and map the result

**Estimated time:** ~20 minutes  
**Dataset:** EuroSAT — 27,000 Sentinel-2 patches, 64×64 px, 13 spectral bands, 10 land cover classes

---

### Sentinel-2 band reference

| Index | Band | Wavelength | What it captures |
|-------|------|-----------|-----------------|
| 0 | B01 | 443 nm | Coastal aerosol |
| 1 | B02 | 490 nm | Blue |
| 2 | B03 | 560 nm | Green |
| 3 | B04 | 665 nm | Red |
| 4 | B05 | 705 nm | Vegetation red edge |
| 5 | B06 | 740 nm | Vegetation red edge |
| 6 | B07 | 783 nm | Vegetation red edge |
| 7 | B08 | 842 nm | Near-infrared (NIR) |
| 8 | B8A | 865 nm | Narrow NIR |
| 9 | B09 | 940 nm | Water vapour |
| 10 | B10 | 1375 nm | SWIR / Cirrus |
| 11 | B11 | 1610 nm | SWIR-1 |
| 12 | B12 | 2190 nm | SWIR-2 |


## Section 0 · Environment Setup

In [ ]:
import os, sys

def get_runtime():
    if 'google.colab' in sys.modules:
        return 'colab'
    elif 'SLURM_JOB_ID' in os.environ or 'KISSKI' in os.environ.get('HPC_NAME', ''):
        return 'hpc'
    else:
        return 'local'

RUNTIME = get_runtime()
print(f"Runtime detected: {RUNTIME}")

if RUNTIME == 'colab':
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DATA_ROOT      = '/content/drive/MyDrive/AI4ES_Seminar/data'
        CHECKPOINT_DIR = '/content/drive/MyDrive/AI4ES_Seminar/checkpoints'
        DOWNLOAD       = False
    except Exception as e:
        print(f"Could not mount Google Drive ({e}). Falling back to local storage.")
        DATA_ROOT      = './data'
        CHECKPOINT_DIR = './checkpoints'
        os.makedirs(DATA_ROOT, exist_ok=True)
        os.makedirs(CHECKPOINT_DIR, exist_ok=True)
        DOWNLOAD       = True
elif RUNTIME == 'hpc':
    # On KISSKI, $HOME is on vast-kisski (~120 GB) — use it for data and checkpoints
    # /opt/conda is a separate system partition that is typically full — never write there
    DATA_ROOT      = os.path.join(os.path.expanduser('~'), 'AI4ES_Seminar', 'data')
    CHECKPOINT_DIR = os.path.join(os.path.expanduser('~'), 'AI4ES_Seminar', 'checkpoints')
    os.makedirs(DATA_ROOT,      exist_ok=True)
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    DOWNLOAD       = True
else:
    DATA_ROOT      = os.path.expanduser('~/AI4ES_Seminar/data')
    CHECKPOINT_DIR = os.path.expanduser('~/AI4ES_Seminar/checkpoints')
    os.makedirs(DATA_ROOT, exist_ok=True)
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    DOWNLOAD       = True

print(f"Data root     : {DATA_ROOT}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")
print(f"Download      : {DOWNLOAD}")


In [ ]:
import subprocess

packages = [
    'torch>=2.0.0', 'torchvision>=0.15.0', 'torchgeo>=0.6.0',
    'numpy', 'matplotlib>=3.7.0',
    'scikit-learn>=1.3.0', 'seaborn>=0.12.0', 'rasterio>=1.3.0'
]

# Try importing first — on KISSKI the conda image may already have everything
try:
    import torch, torchgeo, sklearn, seaborn, rasterio
    print(f'All packages already available (torch {torch.__version__}, torchgeo {torchgeo.__version__})')
except ImportError:
    # --user        : installs into ~/.local, bypassing /opt/conda which is full on KISSKI
    # --no-cache-dir: skips the pip cache to avoid extra disk writes
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--user', '--no-cache-dir'] + packages,
        check=True
    )
    print('Packages installed into ~/.local (--user mode)')


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from torchgeo.datasets import EuroSAT
from torch.utils.data import DataLoader
import random, warnings

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Suppress MPS matmul precision warnings on Apple Silicon
warnings.filterwarnings("ignore", category=RuntimeWarning, message=".*encountered in matmul")

print(f"PyTorch  : {torch.__version__}")
DEVICE = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f"Device   : {DEVICE}")


## Section 1 · Data Loading

In [ ]:
dataset = EuroSAT(
    root=DATA_ROOT,
    split='train',
    bands=EuroSAT.all_band_names,
    download=True
)

print(f"Dataset size : {len(dataset)} samples")
print(f"Classes      : {dataset.classes}")


In [ ]:
sample = dataset[0]
img   = sample['image']
label = sample['label']

print(f"Image shape  : {img.shape}")
print(f"Image dtype  : {img.dtype}")
print(f"Value range  : [{img.min():.1f}, {img.max():.1f}]")
print(f"Label        : {label}  →  {dataset.classes[label]}")


In [ ]:
def get_one_per_class(dataset):
    found, seen_classes = {}, {}
    for i, (path, label) in enumerate(dataset.samples):
        if label not in seen_classes:
            seen_classes[label] = i
        if len(seen_classes) == len(dataset.classes):
            break
    for label, idx in seen_classes.items():
        try:
            s = dataset[idx]
            found[label] = s['image']
        except Exception:
            pass
    return [(found[c], dataset.classes[c]) for c in sorted(found)]

tiles_per_class = get_one_per_class(dataset)
print(f"Collected {len(tiles_per_class)} tiles — one per class:")
for _, cls in tiles_per_class:
    print(f"  · {cls}")


## Section 2 · Band Composites

| Composite | Bands (Sentinel-2) | Tensor indices | Reveals |
|-----------|-------------------|----------------|---------|
| True colour | B04, B03, B02 | [3, 2, 1] | What the human eye sees |
| False colour vegetation | B08, B04, B03 | [7, 3, 2] | Healthy vegetation → bright red |
| False colour urban | B11, B10, B04 | [10, 9, 3] | Built-up areas → cyan/purple |


In [ ]:
def clip_and_normalize(arr, lo_pct=2, hi_pct=98):
    result = np.zeros_like(arr, dtype=np.float32)
    for c in range(arr.shape[2]):
        ch = arr[:, :, c]
        lo, hi = np.percentile(ch, lo_pct), np.percentile(ch, hi_pct)
        result[:, :, c] = np.clip((ch - lo) / (hi - lo + 1e-8), 0, 1)
    return result


In [ ]:
def show_composites(img_tensor, title=''):
    img = img_tensor.numpy().astype(np.float32)   # [13, 64, 64]

    # ── TODO ─────────────────────────────────────────────────────────────────
    # Build three RGB composites using the band index table above.
    # Reminder: tensor indices are 0-based (B11 → idx 10, B10 → idx 9, etc.)
    #
    # true_colour  = clip_and_normalize(np.stack([...], axis=-1))
    # false_veg    = clip_and_normalize(np.stack([...], axis=-1))
    # false_urban  = clip_and_normalize(np.stack([...], axis=-1))
    raise NotImplementedError
    # ─────────────────────────────────────────────────────────────────────────

    fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
    for ax, comp, name in zip(axes,
                               [true_colour, false_veg, false_urban],
                               ['True colour (RGB)', 'False colour: vegetation', 'False colour: urban']):
        ax.imshow(comp)
        ax.set_title(name, fontsize=9)
        ax.axis('off')
    fig.suptitle(title, fontsize=11, y=1.02)
    plt.tight_layout()
    plt.show()


In [ ]:
for cls_name in ['HerbaceousVegetation', 'Residential', 'SeaLake']:
    match = next((img, n) for img, n in tiles_per_class if n == cls_name)
    show_composites(match[0], title=f'Class: {cls_name}')


## Section 3 · Spectral Indices

| Index | Formula | Tensor indices | What it measures |
|-------|---------|----------------|-----------------|
| **NDVI** | (B08 − B04) / (B08 + B04) | (idx7 − idx3) | Vegetation density |
| **NDWI** | (B03 − B08) / (B03 + B08) | (idx2 − idx7) | Open water |
| **NDBI** | (B11 − B08) / (B11 + B08) | (idx10 − idx7) | Built-up density |

> **Note:** Tensor indices are 0-based. B11 = tensor index 10, B08 = tensor index 7, etc.


In [ ]:
def compute_ndvi(img):
    nir = img[7].astype(np.float32)
    red = img[3].astype(np.float32)
    return (nir - red) / (nir + red + 1e-8)


In [ ]:
def compute_ndwi(img):
    # ── TODO ─────────────────────────────────────────────────────────────────
    # NDWI = (Green - NIR) / (Green + NIR)
    # Green = B03 → index ?  |  NIR = B08 → index ?
    raise NotImplementedError
    # ─────────────────────────────────────────────────────────────────────────

def compute_ndbi(img):
    # ── TODO ─────────────────────────────────────────────────────────────────
    # NDBI = (SWIR1 - NIR) / (SWIR1 + NIR)
    # SWIR1 = B11 → tensor index ?  |  NIR = B08 → tensor index ?
    raise NotImplementedError
    # ─────────────────────────────────────────────────────────────────────────


In [ ]:
def show_indices(img_tensor, class_name):
    img = img_tensor.numpy().astype(np.float32)
    ndvi = compute_ndvi(img)
    ndwi = compute_ndwi(img)
    ndbi = compute_ndbi(img)

    fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
    for ax, arr, name, cmap in zip(
        axes, [ndvi, ndwi, ndbi],
        ['NDVI (vegetation)', 'NDWI (water)', 'NDBI (built-up)'],
        ['RdYlGn', 'Blues', 'RdBu_r']
    ):
        im = ax.imshow(arr, cmap=cmap, vmin=-1, vmax=1)
        ax.set_title(name, fontsize=9)
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle(f'Spectral indices — {class_name}', fontsize=11, y=1.02)
    plt.tight_layout()
    plt.show()


In [ ]:
# ── TODO: call show_indices for three contrasting classes ─────────────────────
# Pick classes that you expect to have high NDVI, high NDWI, and high NDBI
# respectively. Use the tiles_per_class list to find them.
raise NotImplementedError


## Section 4 · Unsupervised Clustering

In [ ]:
def cluster_tile(img_tensor, n_clusters=10, seed=42):
    img = img_tensor.numpy().astype(np.float32)
    H, W = img.shape[1], img.shape[2]

    # ── TODO ─────────────────────────────────────────────────────────────────
    # 1. Reshape img from [13, H, W] to [H*W, 13] — one row per pixel
    # 2. Z-score normalise with StandardScaler
    # 3. Fit KMeans(n_clusters, random_state=seed, n_init='auto') and return
    #    the label map reshaped to [H, W]
    raise NotImplementedError
    # ─────────────────────────────────────────────────────────────────────────


In [ ]:
def show_clustering(img_tensor, class_name):
    img = img_tensor.numpy().astype(np.float32)
    rgb = clip_and_normalize(np.stack([img[3], img[2], img[1]], axis=-1))
    cluster_map = cluster_tile(img_tensor)

    fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
    axes[0].imshow(rgb);  axes[0].set_title('True colour', fontsize=9); axes[0].axis('off')
    axes[1].imshow(cluster_map, cmap='tab10', vmin=0, vmax=9)
    axes[1].set_title('K-means clusters (k=10)', fontsize=9); axes[1].axis('off')
    fig.suptitle(f'Class: {class_name}', fontsize=11, y=1.02)
    plt.tight_layout(); plt.show()

for img_t, cls_name in tiles_per_class:
    show_clustering(img_t, cls_name)


## Discussion

1. Which composite was most informative for distinguishing vegetation from built-up?
2. Are NDVI/NDWI values in the expected range for Forest and SeaLake?
3. Does k-means recover the 10 land cover classes cleanly?
4. What does partial spectral separability imply for a supervised model?

